# Data Cleaning & Preprocessing

This notebook uses **train/test only** (EDA and the baseline did not join auxiliary data).
We inspected both splits first and change **only the four issues** that actually show up.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("train", train.shape, "test", test.shape)
print("columns:", list(train.columns))

train (150000, 11) test (50000, 10)
columns: ['RENT_APPROVAL_DATE', 'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'FURNISHED', 'LEASE_COMMENCE_DATE', 'FEE', 'MONTHLY_RENT']


## Quick scan

No missing values, so nothing to impute. The checks below are only for the four issues we will fix.

In [2]:
print("NaNs in train:", int(train.isna().sum().sum()))
print("NaNs in test:", int(test.isna().sum().sum()))

NaNs in train: 0
NaNs in test: 0


### 1. `FLAT_TYPE` — hyphen vs space

Train mixes `3 room` and `3-room`; test only uses the hyphen form.
These are the same category, so we normalise them before encoding.

In [3]:
def normalise_flat_type(s):
    return (
        s.astype(str)
        .str.lower()
        .str.replace("-", " ")
        .str.replace(r"\s+", " ")
        .str.strip()
    )

print("train raw:", sorted(train["FLAT_TYPE"].unique()))
print("test  raw:", sorted(test["FLAT_TYPE"].unique()))
print("train after unify:", sorted(normalise_flat_type(train["FLAT_TYPE"]).unique()))
print("test  after unify:", sorted(normalise_flat_type(test["FLAT_TYPE"]).unique()))

train raw: ['1 room', '1-room', '2 room', '2-room', '3 room', '3-room', '4 room', '4-room', '5 room', '5-room', 'executive']
test  raw: ['1-room', '2-room', '3-room', '4-room', '5-room', 'executive']
train after unify: ['1 room', '2 room', '3 room', '4 room', '5 room', 'executive']
test  after unify: ['1 room', '2 room', '3 room', '4 room', '5 room', 'executive']


### 2. `STREET` — mixed case

`Geylang Bahru` and `geylang bahru` are the same street.
Lowercasing roughly halves the number of labels; `TOWN` / `BLOCK` are already lowercase, so we leave them.

In [4]:
def nunique_raw_vs_lower(s):
    raw = s.nunique()
    lower = s.astype(str).str.strip().str.lower().nunique()
    return raw, lower, raw - lower

for name, df in (("train", train), ("test", test)):
    raw, lower, extra = nunique_raw_vs_lower(df["STREET"])
    print("%s STREET: raw=%d, lowercase=%d, case-only extras=%d" % (name, raw, lower, extra))

train STREET: raw=1146, lowercase=576, case-only extras=570
test STREET: raw=1129, lowercase=573, case-only extras=556


### 3. Exact duplicate rows

Identical training rows add no new information and only double that sample's weight, so we drop extras on **train**.
Test also has copies, but we **keep all 50,000 rows** so the submission order stays aligned.

In [5]:
print("train extra exact dups:", int(train.duplicated().sum()))
print("test  extra exact dups:", int(test.duplicated().sum()), "(keep all rows)")

train extra exact dups: 425
test  extra exact dups: 5508 (keep all rows)


### 4. Constant columns `FURNISHED` and `FEE`

Both columns have a single value on train **and** test, so they cannot help any model.

In [6]:
for col in ("FURNISHED", "FEE"):
    print(col, "train", train[col].unique().tolist(), "test", test[col].unique().tolist())

FURNISHED train ['yes'] test ['yes']
FEE train [0.0] test [0.0]


## Apply the four fixes

Same rules on train and test. Duplicates are dropped on train only, **after** the label fixes so case-only copies become exact duplicates and are removed too.

In [7]:
CONSTANT_COLS = ["FURNISHED", "FEE"]


def clean(df, drop_exact_duplicates=False):
    out = df.copy()
    out["FLAT_TYPE"] = normalise_flat_type(out["FLAT_TYPE"])
    out["STREET"] = out["STREET"].astype(str).str.strip().str.lower()
    out = out.drop(columns=[c for c in CONSTANT_COLS if c in out.columns])
    if drop_exact_duplicates:
        out = out.drop_duplicates(keep="first").reset_index(drop=True)
    return out


train_labelled = clean(train, drop_exact_duplicates=False)
print("train extra exact dups after label fixes:", int(train_labelled.duplicated().sum()))

train_clean = train_labelled.drop_duplicates(keep="first").reset_index(drop=True)
test_clean = clean(test, drop_exact_duplicates=False)

print("train:", train.shape, "->", train_clean.shape)
print("test:", test.shape, "->", test_clean.shape)

train extra exact dups after label fixes: 641
train: (150000, 11) -> (149359, 9)
test: (50000, 10) -> (50000, 8)


## Checks

In [8]:
assert train_clean["FLAT_TYPE"].nunique() == test_clean["FLAT_TYPE"].nunique()
assert set(train_clean["FLAT_TYPE"]) == set(test_clean["FLAT_TYPE"])
assert len(test_clean) == 50000
assert train_clean.isna().sum().sum() == 0
assert test_clean.isna().sum().sum() == 0
assert "FURNISHED" not in train_clean.columns and "FEE" not in train_clean.columns

print("FLAT_TYPE:", sorted(train_clean["FLAT_TYPE"].unique()))
print("STREET unique: train %d, test %d" % (train_clean["STREET"].nunique(), test_clean["STREET"].nunique()))
print("train remaining exact dups:", int(train_clean.duplicated().sum()))
print("ok")

FLAT_TYPE: ['1 room', '2 room', '3 room', '4 room', '5 room', 'executive']
STREET unique: train 576, test 573
train remaining exact dups: 0
ok
